In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import tensorflow as tf
from get_light_status import TrafficLightDetector

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            # Enable memory growth so it doesn't seize the whole card
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(f"Memory growth error: {e}")

TensorFlow GPU Memory Growth Enabled


In [ ]:
detector = YOLO("yolo26m.pt") 
traffic_light_detector = TrafficLightDetector()

In [ ]:
# cap = cv2.VideoCapture(0) # 0 is usually the built-in camera
cap = cv2.VideoCapture('video/hongkong2.mp4')

frame_count = 0

OUTPUT_WIDTH = 1280
OUTPUT_HEIGHT = 720

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame_count += 1

    if frame_count % 10 != 0:
        # display_frame = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT), interpolation=cv2.INTER_AREA)
        # cv2.imshow('Traffic Light Monitor', display_frame) # Keep the window fluid
        # if cv2.waitKey(1) & 0xFF == ord('q'): break
        continue




      # 3. model.track() handles detection + temporal association
    # persist=True tells the tracker to remember IDs from the previous frame
    results = detector.track(frame, classes=[9], persist=True, verbose=False, stream=True)

    for r in results:
        # Check if any boxes were actually detected/tracked
        if r.boxes.id is not None:
            boxes = r.boxes.xyxy.cpu().numpy().astype(int)
            track_ids = r.boxes.id.int().cpu().tolist()
            clss = r.boxes.cls.cpu().tolist()
            confs = r.boxes.conf.cpu().tolist()

            for box, track_id, cls, conf in zip(boxes, track_ids, clss, confs):
                # Annotate or process data
                x1, y1, x2, y2 = box
                
                crop = frame[max(0, y1):y2, max(0, x1):x2]
                
                if crop.size == 0:
                    continue
    
                    # 3. Direct Classification (No resizing unless get_light_state requires it)
                # Tip: Move cv2.resize INSIDE get_light_state only if absolutely necessary
                # color_label = get_light_state(crop)
                state = traffic_light_detector.get_light_state(crop)
                
                # 4. Immediate Drawing (Skips the list/enumerate overhead)
                # bgr_color = COLOR_MAP.get(color_label, (255, 255, 255))
                bgr_color = traffic_light_detector.COLOR_MAP.get(state, (255, 255, 255)) # Default to White if UNKNOWN
     
                cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
                # cv2.putText(frame, f"{color_label}", (x1, y1-10), 
                #         cv2.FONT_HERSHEY_SIMPLEX, 0.9, bgr_color, 2)
                
    display_frame = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT), interpolation=cv2.INTER_AREA)
    cv2.imshow('Traffic Light Monitor', display_frame)
    # Exit on 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
   

cap.release()
cv2.destroyAllWindows()
